# Análisis de sentimiento de reseñas GPU

Este notebook utiliza el modelo `nlptown/bert-base-multilingual-uncased-sentiment` para analizar las reseñas de tarjetas gráficas.

Las reseñas GPU no incluyen valoración individual por reseña, por lo que se analizan en una tabla independiente. El modelo predice una puntuación de 1 a 5 estrellas y después se transforma en sentimiento negativo, neutral o positivo.

## Prueba inicial del modelo

Primero cargamos el modelo y hacemos una prueba sencilla con frases de ejemplo para comprobar que devuelve puntuaciones de 1 a 5 estrellas.

In [1]:
# CELDA 1: CARGA DEL MODELO DE SENTIMIENTO

from transformers import pipeline

MODELO_SENTIMIENTO = "nlptown/bert-base-multilingual-uncased-sentiment"

analizador_sentimiento = pipeline(
    "sentiment-analysis",
    model=MODELO_SENTIMIENTO,
)

analizador_sentimiento

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TextClassificationPipeline: {'model': 'BertForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

## Interpretación del resultado

El modelo devuelve etiquetas en formato `1 star`, `2 stars`, etc. Convertimos esa salida en una puntuación numérica y después en sentimiento.

In [2]:
# CELDA 2: INTERPRETACION DEL RESULTADO DEL MODELO

def interpretar_resultado(resultado):
    estrellas = int(resultado["label"].split()[0])

    if estrellas <= 2:
        sentimiento = "negativo"
    elif estrellas == 3:
        sentimiento = "neutral"
    else:
        sentimiento = "positivo"

    return {
        "estrellas_predichas": estrellas,
        "sentimiento": sentimiento,
        "confianza": resultado["score"],
    }


resenas_prueba = [
    "La tarjeta funciona perfectamente y tiene muy buen rendimiento.",
    "El producto es correcto, pero esperaba algo más por el precio.",
    "Ha llegado con problemas y no funciona como esperaba.",
]

resultados_prueba = analizador_sentimiento(resenas_prueba)

for resena, resultado in zip(resenas_prueba, resultados_prueba):
    print(resena)
    print(interpretar_resultado(resultado))
    print()

La tarjeta funciona perfectamente y tiene muy buen rendimiento.
{'estrellas_predichas': 5, 'sentimiento': 'positivo', 'confianza': 0.7772963643074036}

El producto es correcto, pero esperaba algo más por el precio.
{'estrellas_predichas': 3, 'sentimiento': 'neutral', 'confianza': 0.761410117149353}

Ha llegado con problemas y no funciona como esperaba.
{'estrellas_predichas': 1, 'sentimiento': 'negativo', 'confianza': 0.5816018581390381}



In [3]:
# CELDA 3: CONEXION A POSTGRESQL

import pandas as pd
import psycopg

host = "localhost"
port = "5432"
database = "pccomponentes_ml"
user = "postgres"
password = input("Contraseña PostgreSQL: ")

conn = psycopg.connect(
    host=host,
    port=port,
    dbname=database,
    user=user,
    password=password,
)

print("Conexión creada correctamente")

Conexión creada correctamente


## Muestra de reseñas reales

Cargamos algunas reseñas GPU desde PostgreSQL para revisar el formato de los textos antes de analizar todas las reseñas.

In [4]:
# CELDA 4: MUESTRA DE RESENAS GPU

consulta_muestra_gpu = """
SELECT
    rg.resena_id,
    rg.producto_id,
    p.nombre,
    p.marca,
    rg.fecha_resena_texto,
    rg.texto_resena
FROM resenas_gpu AS rg
INNER JOIN productos AS p
    ON rg.producto_id = p.producto_id
ORDER BY rg.resena_id
LIMIT 10;
"""

with conn.cursor() as cursor:
    cursor.execute(consulta_muestra_gpu)
    filas = cursor.fetchall()
    columnas = [columna.name for columna in cursor.description]

muestra_resenas_gpu = pd.DataFrame(filas, columns=columnas)
muestra_resenas_gpu

,resena_id,producto_id,nombre,marca,fecha_resena_texto,texto_resena
0,gpu_0012_1,gpu_0012,ASRock AMD Radeon RX 7600 Challenger OC 8GB GDDR6,ASRock,Hace 11 meses,Perfecto. Produto ja pronto a usar. Muy buena ...
1,gpu_0012_2,gpu_0012,ASRock AMD Radeon RX 7600 Challenger OC 8GB GDDR6,ASRock,Hace 9 meses,"La tarjeta gráfica es buena para su precio, ¡l..."
2,gpu_0012_3,gpu_0012,ASRock AMD Radeon RX 7600 Challenger OC 8GB GDDR6,ASRock,Hace un mes,Excelente rendimiento por el precio. Lo recomi...
3,gpu_0013_1,gpu_0013,ASRock AMD Radeon RX 7600 Steel Legend OC 8GB ...,ASRock,Hace 11 meses,Perfecto. Produto ja pronto a usar. Muy buena ...
4,gpu_0013_2,gpu_0013,ASRock AMD Radeon RX 7600 Steel Legend OC 8GB ...,ASRock,Hace 9 meses,"La tarjeta gráfica es buena para su precio, ¡l..."
5,gpu_0013_3,gpu_0013,ASRock AMD Radeon RX 7600 Steel Legend OC 8GB ...,ASRock,Hace un mes,Excelente rendimiento por el precio. Lo recomi...
6,gpu_0014_1,gpu_0014,ASRock AMD Radeon RX 7900 XTX Phantom Gaming O...,ASRock,Hace 7 meses,"Muy buena grafica, mucha potencia y color blan..."
7,gpu_0014_2,gpu_0014,ASRock AMD Radeon RX 7900 XTX Phantom Gaming O...,ASRock,Hace un año,Primera gráfica de AMD que pruebo... no me arr...
8,gpu_0014_3,gpu_0014,ASRock AMD Radeon RX 7900 XTX Phantom Gaming O...,ASRock,Hace 10 meses,El producto es nuevo y viene empaquetado. ¡Nun...
9,gpu_0015_1,gpu_0015,ASRock AMD Radeon RX 9060 XT Challenger 16GB O...,ASRock,Hace 5 meses,De lujo funciona perfectamente para el nuevo p...


## Predicción sobre reseñas reales

Aplicamos el modelo a la muestra de reseñas GPU para revisar si las predicciones son coherentes antes de procesar todas las reseñas.

In [5]:
# CELDA 5: PREDICCION SOBRE LA MUESTRA

predicciones_muestra = analizador_sentimiento(
    muestra_resenas_gpu["texto_resena"].tolist(),
    truncation=True,
    max_length=512,
)

predicciones_muestra = [
    interpretar_resultado(resultado)
    for resultado in predicciones_muestra
]

muestra_resenas_gpu["estrellas_predichas"] = [
    resultado["estrellas_predichas"]
    for resultado in predicciones_muestra
]
muestra_resenas_gpu["sentimiento"] = [
    resultado["sentimiento"]
    for resultado in predicciones_muestra
]
muestra_resenas_gpu["confianza"] = [
    resultado["confianza"]
    for resultado in predicciones_muestra
]

muestra_resenas_gpu[
    [
        "resena_id",
        "texto_resena",
        "estrellas_predichas",
        "sentimiento",
        "confianza",
    ]
]

,resena_id,texto_resena,estrellas_predichas,sentimiento,confianza
0,gpu_0012_1,Perfecto. Produto ja pronto a usar. Muy buena ...,5,positivo,0.845630
1,gpu_0012_2,"La tarjeta gráfica es buena para su precio, ¡l...",4,positivo,0.456918
2,gpu_0012_3,Excelente rendimiento por el precio. Lo recomi...,5,positivo,0.693464
3,gpu_0013_1,Perfecto. Produto ja pronto a usar. Muy buena ...,5,positivo,0.845630
4,gpu_0013_2,"La tarjeta gráfica es buena para su precio, ¡l...",4,positivo,0.456918
5,gpu_0013_3,Excelente rendimiento por el precio. Lo recomi...,5,positivo,0.693464
6,gpu_0014_1,"Muy buena grafica, mucha potencia y color blan...",4,positivo,0.544810
7,gpu_0014_2,Primera gráfica de AMD que pruebo... no me arr...,5,positivo,0.463223
8,gpu_0014_3,El producto es nuevo y viene empaquetado. ¡Nun...,5,positivo,0.789438
9,gpu_0015_1,De lujo funciona perfectamente para el nuevo p...,5,positivo,0.752672


## Revisión de resultados

Revisamos el reparto de sentimientos en la muestra antes de lanzar el procesamiento completo.

In [6]:
# CELDA 6: RESUMEN DE RESULTADOS DE LA MUESTRA

muestra_resenas_gpu["sentimiento"].value_counts()

sentimiento
positivo    10
Name: count, dtype: int64

## Procesamiento completo de reseñas GPU

Procesamos todas las reseñas GPU que todavía no tienen resultado de sentimiento y guardamos las predicciones en PostgreSQL.

In [7]:
# CELDA 7: PROCESAMIENTO COMPLETO DE RESENAS GPU

consulta_resenas_pendientes = """
SELECT
    rg.resena_id,
    rg.texto_resena
FROM resenas_gpu AS rg
LEFT JOIN resultados_sentimiento_gpu AS rsg
    ON rg.resena_id = rsg.resena_id
WHERE rsg.resena_id IS NULL
ORDER BY rg.resena_id;
"""

with conn.cursor() as cursor:
    cursor.execute(consulta_resenas_pendientes)
    filas = cursor.fetchall()
    columnas = [columna.name for columna in cursor.description]

resenas_pendientes = pd.DataFrame(filas, columns=columnas)

total_resenas = len(resenas_pendientes)
tamano_lote = 32
resultados_guardados = 0

consulta_insertar_sentimiento = """
INSERT INTO resultados_sentimiento_gpu (
    resena_id,
    estrellas_predichas,
    sentimiento,
    confianza,
    modelo
)
VALUES (
    %(resena_id)s,
    %(estrellas_predichas)s,
    %(sentimiento)s,
    %(confianza)s,
    %(modelo)s
);
"""

for inicio in range(0, total_resenas, tamano_lote):
    lote = resenas_pendientes.iloc[inicio:inicio + tamano_lote]

    predicciones = analizador_sentimiento(
        lote["texto_resena"].tolist(),
        truncation=True,
        max_length=512,
    )

    resultados_lote = []

    for resena_id, prediccion in zip(lote["resena_id"], predicciones):
        resultado = interpretar_resultado(prediccion)

        resultados_lote.append(
            {
                "resena_id": resena_id,
                "estrellas_predichas": resultado["estrellas_predichas"],
                "sentimiento": resultado["sentimiento"],
                "confianza": resultado["confianza"],
                "modelo": MODELO_SENTIMIENTO,
            }
        )

    with conn.cursor() as cursor:
        cursor.executemany(consulta_insertar_sentimiento, resultados_lote)

    conn.commit()

    resultados_guardados += len(resultados_lote)

    print(
        f"Guardadas {resultados_guardados} de {total_resenas} reseñas pendientes"
    )

print(f"Proceso terminado. Resultados nuevos guardados: {resultados_guardados}")

Guardadas 32 de 3564 reseñas pendientes
Guardadas 64 de 3564 reseñas pendientes
Guardadas 96 de 3564 reseñas pendientes
Guardadas 128 de 3564 reseñas pendientes
Guardadas 160 de 3564 reseñas pendientes
Guardadas 192 de 3564 reseñas pendientes
Guardadas 224 de 3564 reseñas pendientes
Guardadas 256 de 3564 reseñas pendientes
Guardadas 288 de 3564 reseñas pendientes
Guardadas 320 de 3564 reseñas pendientes
Guardadas 352 de 3564 reseñas pendientes
Guardadas 384 de 3564 reseñas pendientes
Guardadas 416 de 3564 reseñas pendientes
Guardadas 448 de 3564 reseñas pendientes
Guardadas 480 de 3564 reseñas pendientes
Guardadas 512 de 3564 reseñas pendientes
Guardadas 544 de 3564 reseñas pendientes
Guardadas 576 de 3564 reseñas pendientes
Guardadas 608 de 3564 reseñas pendientes
Guardadas 640 de 3564 reseñas pendientes
Guardadas 672 de 3564 reseñas pendientes
Guardadas 704 de 3564 reseñas pendientes
Guardadas 736 de 3564 reseñas pendientes
Guardadas 768 de 3564 reseñas pendientes
Guardadas 800 de 35

## Resultado final

El análisis de sentimiento de reseñas GPU se ha completado y todos los resultados se han guardado en PostgreSQL.

In [8]:
# CELDA 8: COMPROBACION FINAL

consulta_resumen_final = """
SELECT
    sentimiento,
    COUNT(*) AS total
FROM resultados_sentimiento_gpu
GROUP BY sentimiento
ORDER BY sentimiento;
"""

resumen_sentimiento_gpu = pd.read_sql(consulta_resumen_final, conn)
resumen_sentimiento_gpu

C:\Users\raulr\AppData\Local\Temp\ipykernel_24672\4133161782.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resumen_sentimiento_gpu = pd.read_sql(consulta_resumen_final, conn)


,sentimiento,total
0,negativo,422
1,neutral,406
2,positivo,2736
